# GlassDesign ML — самостоятельный запуск в Google Colab
Бесплатная демонстрация программы на оригинальных четырех обученных моделях и GlassNet. **Это не постоянно работающий сайт.** Ссылка на интерфейс временная и существует только пока активен Colab запускающего пользователя.

**Порядок запуска:** откройте блокнот в Colab; запустите все ячейки; после установки зависимостей и загрузки моделей получите временную ссылку.

Модели `.joblib` из доверенного источника могут выполнять код при десериализации. Используйте только опубликованные автором файлы.


In [ ]:
# ID ЧЕТЫРЁХ МОДЕЛЕЙ В GOOGLE DRIVE (вставляет автор перед публикацией)
MODEL_FILE_IDS = {
    'density_extra_trees_final.joblib': '',
    'refractive_index_extra_trees_final_v2.joblib': '',
    'rf_tg_final.joblib': '',
    'extra_trees_softening_final.joblib': '',
}
assert all(str(v).strip() for v in MODEL_FILE_IDS.values()), 'Ссылки на модели ещё не добавлены автором'


In [ ]:
import sys, subprocess, pathlib
project=pathlib.Path('/content/glassdesign-ml')
if not project.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/polinashidlovskaya/glassdesign-ml.git',str(project)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(project/'requirements.txt'),'gdown'],check=True)
import gdown
dest=project/'runtime'/'GlassDesignML'/'models'
dest.mkdir(parents=True,exist_ok=True)
for filename,file_id in MODEL_FILE_IDS.items():
    file=dest/filename
    if not file.exists() or file.stat().st_size<1024*1024:
        gdown.download(id=str(file_id).strip(),output=str(file),quiet=False)
    if not file.exists() or file.stat().st_size<1024*1024:
        raise RuntimeError('Ошибка загрузки модели: '+filename)
    print('✅',filename,round(file.stat().st_size/1024**2,1),'МиБ')
print('✅ Четыре модели готовы')


In [ ]:
import os,time,subprocess,pathlib,re,requests
project=pathlib.Path('/content/glassdesign-ml')
port='8501'
app_log=open('/content/glassdesign_streamlit.log','w')
app=subprocess.Popen(['streamlit','run','streamlit_app.py','--server.port='+port,'--server.address=0.0.0.0','--server.headless=true','--browser.gatherUsageStats=false'],cwd=str(project),stdout=app_log,stderr=subprocess.STDOUT)
for _ in range(60):
    if app.poll() is not None:
        raise RuntimeError('Streamlit завершился. Откройте /content/glassdesign_streamlit.log')
    try:
        if requests.get('http://127.0.0.1:'+port+'/_stcore/health',timeout=1).ok:break
    except requests.RequestException:pass
    time.sleep(1)
else:raise RuntimeError('Streamlit не запустился за 60 секунд')
cf=pathlib.Path('/content/cloudflared')
if not cf.exists():
    data=requests.get('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',timeout=120)
    data.raise_for_status(); cf.write_bytes(data.content); cf.chmod(0o755)
tunnel_log=pathlib.Path('/content/glassdesign_tunnel.log')
tl=tunnel_log.open('w')
tunnel=subprocess.Popen([str(cf),'tunnel','--no-autoupdate','--url','http://127.0.0.1:'+port],stdout=tl,stderr=subprocess.STDOUT)
public_url=None
for _ in range(75):
    time.sleep(1)
    match=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',tunnel_log.read_text(errors='replace'))
    if match:
        public_url=match.group(0);break
if not public_url:
    raise RuntimeError('Туннель не запустился; см. /content/glassdesign_tunnel.log')
print('\n✅ GLASSDESIGN ML ОТКРЫТ')
print('Временная ссылка:',public_url)
print('ОСТАВЬТЕ GOOGLE COLAB ВКЛЮЧЕННЫМ, ПОКА ПОЛЬЗУЕТЕСЬ САЙТОМ.')
print('Для повторного запуска в другой день нужно выполнить этот блокнот снова.')
